In [1]:

import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

# ── Configuration ──────────────────────────────────────────────────────────────

NEO4J_URI      = os.getenv("NEO4J_URI",      "bolt://localhost:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "Zensar@123")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

# ── Models ─────────────────────────────────────────────────────────────────────

llm        = ChatOllama(model="llama3.2", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# ── Neo4j ──────────────────────────────────────────────────────────────────────

print("Connecting to Neo4j...")
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)
graph.refresh_schema()
print("Connected.\n")

# ── Prompts ────────────────────────────────────────────────────────────────────

CYPHER_GENERATION_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a Neo4j Cypher expert. Given the graph schema, write a Cypher
query that retrieves data needed to answer the question.
Only use node labels and relationship types present in the schema.
Always match entity names case-insensitively using toLower(), for example:
  WHERE toLower(n.id) = toLower('OpenAI')
Return ONLY the Cypher query, no explanation.

Schema:
{schema}""",
    ),
    ("human", "Question: {question}"),
])

FINAL_ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful AI assistant answering questions about an AI ecosystem.
You have two sources of retrieved information:

GRAPH DATA (entities and relationships from Neo4j):
{graph_context}

VECTOR DATA (semantically similar text passages):
{vector_context}

Use both sources to give a comprehensive answer.
Prefer graph data for precise entity/relationship facts.
Use vector data for broader context and explanations.""",
    ),
    ("human", "Question: {question}"),
])

# ── Graph retrieval ────────────────────────────────────────────────────────────

cypher_chain = CYPHER_GENERATION_PROMPT | llm | StrOutputParser()


def retrieve_from_graph(question: str) -> tuple[str, str]:
    """Ask LLM to generate a Cypher query, run it, return (cypher, results)."""
    cypher_query = cypher_chain.invoke({
        "schema": graph.schema,
        "question": question,
    })

    # Strip markdown code fences if the LLM wrapped the query
    cypher_query = (
        cypher_query.strip()
        .removeprefix("```cypher")
        .removeprefix("```")
        .removesuffix("```")
        .strip()
    )

    try:
        results = graph.query(cypher_query)
        results_str = "\n".join(str(row) for row in results) if results else "No results found."
    except Exception as e:
        results_str = f"Cypher execution error: {e}"

    return cypher_query, results_str


# ── Vector retrieval ───────────────────────────────────────────────────────────

def get_vector_retriever():
    try:
        store = Neo4jVector.from_existing_index(
            embedding=embeddings,
            url=NEO4J_URI,
            username=NEO4J_USERNAME,
            password=NEO4J_PASSWORD,
            database=NEO4J_DATABASE,
            index_name="graphrag_vector_index",
            node_label="Document",
            text_node_property="text",
            embedding_node_property="embedding",
        )
        print("Loaded existing vector index.\n")
    except Exception:
        print("Building vector index from Document nodes...\n")
        store = Neo4jVector.from_existing_graph(
            embedding=embeddings,
            url=NEO4J_URI,
            username=NEO4J_USERNAME,
            password=NEO4J_PASSWORD,
            database=NEO4J_DATABASE,
            index_name="graphrag_vector_index",
            node_label="Document",
            text_node_properties=["text"],
            embedding_node_property="embedding",
        )
    return store.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.5},
    )


def retrieve_from_vector(question: str, retriever) -> list[str]:
    try:
        docs = retriever.invoke(question)
        return [doc.page_content for doc in docs]
    except Exception as e:
        return [f"Vector retrieval error: {e}"]


# ── Main GraphRAG query function ───────────────────────────────────────────────

def graph_rag_query(question: str, retriever) -> str:
    print("\n" + "=" * 60)
    print(f"INPUT QUERY: {question}")
    print("=" * 60)

    # Step 1 — LLM generates and runs Cypher
    print("\n[Step 1] Generating Cypher query from question...")
    cypher_query, graph_results = retrieve_from_graph(question)
    print(f"\n  Generated Cypher:\n  {cypher_query}")
    print(f"\n  Graph results:\n  {graph_results}")

    # Step 2 — Vector retrieval
    print("\n[Step 2] Running vector similarity search...")
    vector_chunks = retrieve_from_vector(question, retriever)
    print(f"  Retrieved {len(vector_chunks)} chunk(s):")
    for i, chunk in enumerate(vector_chunks, 1):
        print(f"  [{i}] {chunk[:120]}...")

    # Step 3 — Single LLM call with combined context
    graph_ctx  = graph_results if graph_results else "No graph data found."
    vector_ctx = "\n\n---\n\n".join(vector_chunks) if vector_chunks else "No vector data found."

    print("\n[Step 3] Sending combined context to LLM...")
    print(f"  - Graph context  ({len(graph_ctx)} chars)")
    print(f"  - Vector context ({len(vector_ctx)} chars)")

    final_chain = FINAL_ANSWER_PROMPT | llm | StrOutputParser()
    return final_chain.invoke({
        "question": question,
        "graph_context": graph_ctx,
        "vector_context": vector_ctx,
    })


# ── Interactive loop ───────────────────────────────────────────────────────────

if __name__ == "__main__":
    retriever = get_vector_retriever()

    print("=" * 60)
    print("GraphRAG Q&A (Cypher approach)  |  type 'quit' to exit")
    print("=" * 60 + "\n")

    while True:
        try:
            user_input = input("Ask a question: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nExiting.")
            break

        if not user_input:
            continue
        if user_input.lower() in {"quit", "exit", "q"}:
            print("Goodbye!")
            break

        answer = graph_rag_query(user_input, retriever)

        print("\n" + "─" * 60)
        print("FINAL ANSWER:")
        print("─" * 60)
        print(answer)
        print("─" * 60 + "\n")

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connecting to Neo4j...
Connected.

Building vector index from Document nodes...

GraphRAG Q&A (Cypher approach)  |  type 'quit' to exit


INPUT QUERY: How Google DeepMind was formed?

[Step 1] Generating Cypher query from question...

  Generated Cypher:
  MATCH (o:Organization {id: 'GoogleDeepMind'})
OPTIONAL MATCH (o)-[:FOUNDING_MEMBER]->(p:Person)
RETURN p, o

  Graph results:
  No results found.

[Step 2] Running vector similarity search...


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k RETURN reduce(str='', k IN ['text'] | str + '\\n' + k + ': ' + coalesce(node[k], '')) AS text, node {.*, `embedding`: Null, `text`: Null} AS metadata, score"


  Retrieved 3 chunk(s):
  [1] 
text: Google DeepMind was formed by the merger of Google Brain and DeepMind in 2023. Google DeepMind is led by Demis Ha...
  [2] 
text: Mistral AI was founded in 2023 by Arthur Mensch, Guillaume Lample, and Timothée Lacroix, who previously worked at...
  [3] 
text: Artificial Intelligence Research Ecosystem

OpenAI was founded in 2015 by Sam Altman, Elon Musk, Greg Brockman, I...

[Step 3] Sending combined context to LLM...
  - Graph context  (17 chars)
  - Vector context (2450 chars)

────────────────────────────────────────────────────────────
FINAL ANSWER:
────────────────────────────────────────────────────────────
According to the GRAPH DATA, I couldn't find any information on the formation of Google DeepMind. However, I can provide more context using the VECTOR DATA.

Google DeepMind was formed by the merger of Google Brain and DeepMind in 2023. This merger brought together the expertise of both organizations to create a leading AI research and deve